# Filter AMR papers with LLM

Use the `llm.py` helper to ask a model whether a paper's abstract asserts that a pathogen is showing **significant resistance** to a given antibiotic.


In [5]:

# --- LLM filtering configuration ---
import pandas as pd

from amr.llm import create_llm_client
from amr.filtering import (
    DEFAULT_SYSTEM_PROMPT,
    prepare_pmids_to_review,
    run_llm_filter,
)
from amr.migrations import backfill_filtered_to_combination_keys

SYSTEM_PROMPT = """
You are an expert microbiologist reviewing evidence. Given a title and abstract, decide whether the authors explicitly state or conclude that the specified pathogen is showing clinically meaningful resistance to the specified antibiotic.

Rules:
- Answer "yes" only if the abstract clearly reports resistance, reduced susceptibility, treatment failure due to resistance, or emergence of resistant isolates for that pathogen/antibiotic pair.
- Answer "no" if the abstract does not make that claim, is about a different organism/drug, only mentions testing without resistance, or only provides background.
- Answer "uncertain" if the abstract is missing, ambiguous, or only indirectly related.
- Base the decision only on the provided text; do not infer beyond it.

Return compact JSON with keys: decision (yes/no/uncertain) and rationale (one short sentence).
"""

LLM_PROVIDER = "openai"
MODEL_NAME = "gpt-4o-mini"
NCBI_EMAIL = "your_email@example.com"
NEW_PMIDS_TO_PROCESS = 100000  # number of new PMIDs to process in this run (None for all)
N_PMIDS_PER_RUN = 100  # how many unprocessed PMIDs to handle per run
RATE_LIMIT = 0.35  # seconds between PubMed calls
OUTPUT_PATH = "pmids_results_filtered.parquet"
LOG_PATH = "pmid_filter.log"
MAX_LLM_CALLS = None  # set to an integer to cap spend per run
DRY_RUN = False  # set True to skip LLM calls
SAVE_EVERY = 50  # checkpoint frequency

llm_kwargs = {}
if MODEL_NAME:
    llm_kwargs["model"] = MODEL_NAME

llm = create_llm_client(provider=LLM_PROVIDER, **llm_kwargs)
ABSTRACT_CACHE_PATH = "pmid_abstracts.parquet"


In [6]:

import os
import pandas as pd
from amr.filtering import prepare_pmids_to_review

# One-time migration to combination-level rows (safe no-op if already done)
try:
    before, after = backfill_filtered_to_combination_keys(OUTPUT_PATH, "pmids_results.parquet")
    if before and before != after:
        print(f"Migrated filtered results to combination-level: {before} -> {after}")
except Exception:
    pass

# Load previous decisions (combination-level: pmid+pathogen+antibiotic)
if os.path.exists(OUTPUT_PATH):
    reviewed_df = pd.read_parquet(OUTPUT_PATH)
    if not reviewed_df.empty and {"pmid", "pathogen", "antibiotic"}.issubset(reviewed_df.columns):
        reviewed_keys = set(
            zip(
                reviewed_df["pmid"].astype(str),
                reviewed_df["pathogen"].astype(str),
                reviewed_df["antibiotic"].astype(str),
            )
        )
    else:
        # Fallback: only PMID column present
        reviewed_keys = set((str(p), "", "") for p in reviewed_df["pmid"].astype(str)) if not reviewed_df.empty else set()
    print(f"Resuming with {len(reviewed_keys):,} combination-level entries already reviewed from {OUTPUT_PATH}")
else:
    reviewed_df = None
    reviewed_keys = set()

initial_reviewed_count = len(reviewed_keys)

pmid_df = pd.read_parquet("pmids_results.parquet")
# Build full exploded table to report totals
full_long_df = prepare_pmids_to_review(pmid_df, exclude_pmids=None)
full_total = len(full_long_df)

# Exclude reviewed first, THEN cap to the next chunk
long_df = full_long_df
if reviewed_keys:
    keys_series = (
        long_df["pmid"].astype(str) + "|" + long_df["pathogen"].astype(str) + "|" + long_df["antibiotic"].astype(str)
    )
    keys_to_exclude = set("|".join(k) if isinstance(k, tuple) else str(k) for k in reviewed_keys)
    long_df = long_df[~keys_series.isin(keys_to_exclude)]

pmid_total = len(long_df)
reviewed_count = len(reviewed_keys)
print(f"Total PMIDs in source (exploded): {full_total:,}")
print(f"Loaded {pmid_total:,} candidate PMIDs after exclusion; reviewed set size: {reviewed_count:,}")
if NEW_PMIDS_TO_PROCESS is None:
    print("Planning to process all remaining PMIDs this run.")
else:
    print(f"Planning to process up to {NEW_PMIDS_TO_PROCESS:,} new PMIDs this run.")


Resuming with 243,333 combination-level entries already reviewed from pmids_results_filtered.parquet
Total PMIDs in source (exploded): 518,057
Loaded 274,724 candidate PMIDs after exclusion; reviewed set size: 243,333
Planning to process up to 100,000 new PMIDs this run.


In [ ]:

%%time

import time
from collections import deque
start_ts = time.time()
# Track recent throughput to compute ETA based on recent runtime only
RECENT_WINDOW_SEC = 900  # 15 minutes
recent_progress = deque()  # (timestamp, processed_so_far)
recent_rate = None  # items/sec over recent window
# Compute target_total for ETA based on current queue (combination-level) and any quota
keys_series_all = (
    long_df["pmid"].astype(str) + "|" + long_df["pathogen"].astype(str) + "|" + long_df["antibiotic"].astype(str)
)
keys_exclude_all = set("|".join(k) if isinstance(k, tuple) else str(k) for k in reviewed_keys)
base_remaining_df = long_df[~keys_series_all.isin(keys_exclude_all)]
target_total = len(base_remaining_df) if NEW_PMIDS_TO_PROCESS is None else min(NEW_PMIDS_TO_PROCESS, len(base_remaining_df))

iter_idx = 0

while True:
    # Compute global remaining before applying any per-run quota
    keys_series_loop = (
        long_df["pmid"].astype(str) + "|" + long_df["pathogen"].astype(str) + "|" + long_df["antibiotic"].astype(str)
    )
    keys_exclude_loop = set("|".join(k) if isinstance(k, tuple) else str(k) for k in reviewed_keys)
    remaining_df = long_df[~keys_series_loop.isin(keys_exclude_loop)]
    remaining_all = len(remaining_df)
    processed_this_run = len(reviewed_keys) - initial_reviewed_count

    if NEW_PMIDS_TO_PROCESS is not None:
        remaining_quota = NEW_PMIDS_TO_PROCESS - processed_this_run
        if remaining_quota <= 0:
            print(f"Reached NEW_PMIDS_TO_PROCESS={NEW_PMIDS_TO_PROCESS}; stopping.")
            break
        remaining_df = remaining_df.head(remaining_quota)

    if remaining_df.empty:
        print("No PMIDs left to process.")
        break

    batch_df = remaining_df.head(N_PMIDS_PER_RUN)
    remaining_after_batch = max(len(remaining_df) - len(batch_df), 0)  # per-run queued after this batch
    global_remaining_after_batch = max(remaining_all - len(batch_df), 0)  # total remaining after this batch
    iter_idx += 1
    processed_so_far = max(0, len(reviewed_keys) - initial_reviewed_count)
    remaining_target = max(0, target_total - processed_so_far)

    # Compute ETAs using only recent runtime throughput; ignore idle delays
    rate = recent_rate if (recent_rate is not None and recent_rate > 0) else None
    if rate:
        eta_run_sec = int(remaining_after_batch / rate)
        eta_total_sec = int(global_remaining_after_batch / rate)
        # Convert to days, hours, minutes
        eta_run_d, rem = divmod(eta_run_sec, 86400)
        eta_run_h, rem = divmod(rem, 3600)
        eta_run_m, _ = divmod(rem, 60)
        eta_total_d, remt = divmod(eta_total_sec, 86400)
        eta_total_h, remt = divmod(remt, 3600)
        eta_total_m, _ = divmod(remt, 60)
        eta_str = f"[ETA {eta_run_d}d {eta_run_h}h {eta_run_m}m | total {eta_total_d}d {eta_total_h}h {eta_total_m}m]"
    else:
        eta_str = "[ETA -- | total --]"
    print(
        f"Iteration {iter_idx}: processing {len(batch_df):,} PMIDs; {remaining_after_batch:,} still queued {eta_str}"
    )

    filtered_df = run_llm_filter(
        batch_df,
        llm,
        email=NCBI_EMAIL,
        rate_limit=RATE_LIMIT,
        save_every=SAVE_EVERY,
        system_prompt=SYSTEM_PROMPT,
        progress_every=10,
        resume_from=reviewed_df,
        seen_pmids=reviewed_keys,
        output_path=OUTPUT_PATH,
        log_path=LOG_PATH,
        max_llm_calls=MAX_LLM_CALLS,
        dry_run=DRY_RUN,
        abstract_cache_path=ABSTRACT_CACHE_PATH,
    )

    reviewed_df = filtered_df
    if not reviewed_df.empty:
        reviewed_keys = set(
            zip(
                reviewed_df["pmid"].astype(str),
                reviewed_df["pathogen"].astype(str),
                reviewed_df["antibiotic"].astype(str),
            )
        )
    else:
        reviewed_keys = set()

    # Build positive-only dataset and save
    pmid_df_full = pd.read_parquet("pmids_results.parquet")

    # Merge decisions onto exploded PMIDs at combination level
    positive_flags = filtered_df[filtered_df["decision"].str.lower() == "yes"][
        ["pmid", "pathogen", "antibiotic", "start_date", "end_date"]
    ].copy()
    positive_flags["pmid"] = positive_flags["pmid"].astype(str)
    positive_rows = (
        pmid_df_full.explode("pmids")
        .assign(pmid=lambda d: d["pmids"].astype(str).str.extract(r"(\d+)")[0])
        .dropna(subset=["pmid"])
        .merge(
            positive_flags,
            on=["pmid", "pathogen", "antibiotic", "start_date", "end_date"],
            how="inner",
        )
    )

    # Reassemble to original grain
    pmids_positive = (
        positive_rows.groupby(["pathogen", "antibiotic", "start_date", "end_date"], as_index=False)
        .agg(
            pmids=("pmid", list),
            count=("pmid", "size"),
            venue_histogram=("venue_histogram", "first"),
        )
    )

    positive_path = "pmids_results_positive.parquet"
    pmids_positive.to_parquet(positive_path, index=False)
    print(f"Saved positive-only dataset to {positive_path} with {len(pmids_positive)} rows")

    # Update recent progress window for ETA (use post-batch progress)
    now = time.time()
    processed_after = max(0, len(reviewed_keys) - initial_reviewed_count)
    recent_progress.append((now, processed_after))
    # Drop samples outside the recent window
    cutoff = now - RECENT_WINDOW_SEC
    while len(recent_progress) > 1 and recent_progress[0][0] < cutoff:
        recent_progress.popleft()
    # Recompute recent rate from window
    if len(recent_progress) >= 2:
        dt = recent_progress[-1][0] - recent_progress[0][0]
        dp = recent_progress[-1][1] - recent_progress[0][1]
        if dt > 0 and dp > 0:
            recent_rate = dp / dt


Iteration 1: processing 100 PMIDs; 99,900 still queued [ETA -- | total --]
Processed  10/100  no:      8  yes:      2  uncertain:      0  |  cumulative  243343  no: 162699  yes:  73669  uncertain:   6975
Processed  20/100  no:     18  yes:      2  uncertain:      0  |  cumulative  243353  no: 162709  yes:  73669  uncertain:   6975
Processed  30/100  no:     28  yes:      2  uncertain:      0  |  cumulative  243363  no: 162719  yes:  73669  uncertain:   6975
Processed  40/100  no:     38  yes:      2  uncertain:      0  |  cumulative  243373  no: 162729  yes:  73669  uncertain:   6975
Processed  50/100  no:     48  yes:      2  uncertain:      0  |  cumulative  243383  no: 162739  yes:  73669  uncertain:   6975
Processed  60/100  no:     57  yes:      3  uncertain:      0  |  cumulative  243393  no: 162748  yes:  73670  uncertain:   6975
Processed  70/100  no:     67  yes:      3  uncertain:      0  |  cumulative  243403  no: 162758  yes:  73670  uncertain:   6975
Processed  80/100  no: